# 05 - Evaluate Multilingual Model

This notebook evaluates trained models and compares experiments.

**Prerequisites:** Run notebook 04 first.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil')
print('Working directory:', os.getcwd())

Working directory: /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil


In [3]:
!pip install -q torch transformers scikit-learn pandas numpy pyyaml

In [4]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

login(token=HF_TOKEN)

print("Hugging Face authentication successful.")

Hugging Face authentication successful.


In [5]:
# Evaluate Experiment C model
!python scripts/evaluate_multilingual.py --model-dir models/indicbert-v3-270m_en_ta_enta

Device: cuda
Loading model from /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil/models/indicbert-v3-270m_en_ta_enta...
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}
Loading weights: 100% 237/237 [00:05<00:00, 41.28it/s]
Model loaded successfully.
Loaded test set: 5359 samples

Evaluating EN (2165 samples)...
  Accuracy:    0.8185
  Macro F1:    0.7658
  Weighted F1: 0.8191

              precision    recall  f1-score   support

    negative       0.90      0.89      0.89      1363
     neutral       0.65      0.65      0.65       459
    positive       0.74      0.77      0.76       343

    accuracy                           0.82      2165
   macro avg       0.76      0.77      0.77      2165
weighted avg       0.82      0.82      0.82      2165


Evaluating EN-TA (1029 samples)...


In [6]:
# Compare all experiments
import json
import pandas as pd

experiments = {}
results_dir = 'results'
for exp_dir in os.listdir(results_dir):
    results_path = os.path.join(results_dir, exp_dir, 'results.json')
    if os.path.exists(results_path):
        with open(results_path) as f:
            experiments[exp_dir] = json.load(f)

if experiments:
    rows = []
    for name, data in experiments.items():
        for lang, metrics in data.get('per_language', {}).items():
            rows.append({
                'Experiment': name,
                'Language': lang,
                'Accuracy': metrics.get('accuracy', 0),
                'Macro F1': metrics.get('macro_f1', 0),
            })
    comparison_df = pd.DataFrame(rows)
    print(comparison_df.to_string(index=False))
else:
    print('No experiment results found yet.')

                  Experiment Language  Accuracy  Macro F1
        indicbert-v3-270m_en       en  0.837875  0.786334
        indicbert-v3-270m_en       ta  0.780600  0.724192
        indicbert-v3-270m_en    en-ta  0.783285  0.706421
     indicbert-v3-270m_en_ta       en  0.836028  0.778128
     indicbert-v3-270m_en_ta       ta  0.811085  0.751725
     indicbert-v3-270m_en_ta    en-ta  0.779397  0.696218
indicbert-v3-270m_en_ta_enta       en  0.818014  0.765164
indicbert-v3-270m_en_ta_enta       ta  0.801386  0.738364
indicbert-v3-270m_en_ta_enta    en-ta  0.791059  0.721255
 xlm-roberta-base_en_ta_enta       en  0.835104  0.784440
 xlm-roberta-base_en_ta_enta       ta  0.813857  0.755551
 xlm-roberta-base_en_ta_enta    en-ta  0.799806  0.725372


In [7]:
# Generate translation quality annotation template
!python scripts/evaluate_translation_quality.py --generate --samples 200

Loaded 14427 Tamil translations

[OK] Annotation template saved: /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil/data/translation_quality_template.csv
   Samples: 200

   Distribution:
     negative: 67
     neutral: 67
     positive: 66

Instructions:
  1. Open /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil/data/translation_quality_template.csv in a spreadsheet editor
  2. For each row, fill in:
     - translation_correct: Yes/No
     - meaning_preserved: Yes/No
     - sentiment_preserved: Yes/No
     - natural_tamil: Yes/No/Partial
     - notes: any additional observations
  3. Save the file
  4. Run: python scripts/evaluate_translation_quality.py --calculate --input /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil/data/translation_quality_template.csv
